# Practical No. 03: Autoencoder and Variational Autoencoder (VAE)

## Aim
Build an **Autoencoder** and a **Variational Autoencoder (VAE)** for image reconstruction and generation. Compare both models by analyzing the quality of reconstructed and generated images.

Dataset: **MNIST handwritten digits**


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

# Load MNIST dataset
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()

# Normalize pixel values to 0-1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Add channel dimension
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print("Training shape:", x_train.shape)
print("Testing shape:", x_test.shape)


## 1. Autoencoder for Image Reconstruction

In [ ]:
from tensorflow.keras import layers, Model

latent_dim = 32

# Encoder
encoder_inputs = layers.Input(shape=(28, 28, 1))
x = layers.Flatten()(encoder_inputs)
x = layers.Dense(128, activation="relu")(x)
latent = layers.Dense(latent_dim, activation="relu", name="latent_space")(x)

encoder = Model(encoder_inputs, latent, name="encoder")

# Decoder
decoder_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(128, activation="relu")(decoder_inputs)
x = layers.Dense(28 * 28, activation="sigmoid")(x)
decoder_outputs = layers.Reshape((28, 28, 1))(x)

decoder = Model(decoder_inputs, decoder_outputs, name="decoder")

# Complete Autoencoder
autoencoder_inputs = encoder_inputs
encoded = encoder(autoencoder_inputs)
decoded = decoder(encoded)

autoencoder = Model(autoencoder_inputs, decoded, name="autoencoder")

autoencoder.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

autoencoder.summary()


In [ ]:
# Train Autoencoder
history_ae = autoencoder.fit(
    x_train, x_train,
    validation_data=(x_test, x_test),
    epochs=10,
    batch_size=256,
    verbose=1
)


In [ ]:
# Evaluate Autoencoder
ae_loss, ae_accuracy = autoencoder.evaluate(x_test, x_test, verbose=0)
print("Autoencoder Test Loss:", ae_loss)
print("Autoencoder Test Accuracy:", ae_accuracy)


In [ ]:
# Display original and reconstructed images
reconstructed_ae = autoencoder.predict(x_test[:10], verbose=0)

plt.figure(figsize=(15, 4))

for i in range(10):
    # Original
    plt.subplot(2, 10, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.title("Original")

    # Reconstruction
    plt.subplot(2, 10, i + 11)
    plt.imshow(reconstructed_ae[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.title("Reconstructed")

plt.tight_layout()
plt.show()


## 2. Variational Autoencoder (VAE) for Reconstruction and Generation

In [ ]:
latent_dim = 2

# Sampling layer for VAE
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# VAE Encoder
vae_encoder_inputs = layers.Input(shape=(28, 28, 1))
x = layers.Flatten()(vae_encoder_inputs)
x = layers.Dense(256, activation="relu")(x)

z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])

vae_encoder = Model(
    vae_encoder_inputs,
    [z_mean, z_log_var, z],
    name="vae_encoder"
)

# VAE Decoder
vae_decoder_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(256, activation="relu")(vae_decoder_inputs)
x = layers.Dense(28 * 28, activation="sigmoid")(x)
vae_decoder_outputs = layers.Reshape((28, 28, 1))(x)

vae_decoder = Model(vae_decoder_inputs, vae_decoder_outputs, name="vae_decoder")


In [ ]:
# Custom VAE model
class VAE(Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = tf.keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker
        ]

    def train_step(self, data):
        if isinstance(data, tuple):
            data = data[0]

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)

            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    tf.keras.losses.binary_crossentropy(data, reconstruction),
                    axis=(1, 2)
                )
            )

            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                    axis=1
                )
            )

            total_loss = reconstruction_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result()
        }

vae = VAE(vae_encoder, vae_decoder)
vae.compile(optimizer=tf.keras.optimizers.Adam())


In [ ]:
# Train VAE
history_vae = vae.fit(
    x_train,
    epochs=10,
    batch_size=256,
    validation_data=(x_test, None),
    verbose=1
)


In [ ]:
# Reconstruct images using VAE
z_mean, z_log_var, z = vae_encoder.predict(x_test[:10], verbose=0)
reconstructed_vae = vae_decoder.predict(z, verbose=0)

plt.figure(figsize=(15, 4))

for i in range(10):
    plt.subplot(2, 10, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.title("Original")

    plt.subplot(2, 10, i + 11)
    plt.imshow(reconstructed_vae[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.title("VAE Reconstruction")

plt.tight_layout()
plt.show()


## 3. Image Generation using VAE

In [ ]:
# Generate new images by sampling random points from latent space
n = 10
random_latent_vectors = np.random.normal(size=(n, latent_dim))
generated_images = vae_decoder.predict(random_latent_vectors, verbose=0)

plt.figure(figsize=(15, 2))

for i in range(n):
    plt.subplot(1, n, i + 1)
    plt.imshow(generated_images[i].squeeze(), cmap="gray")
    plt.axis("off")

plt.suptitle("New Images Generated by VAE")
plt.tight_layout()
plt.show()


## 4. Performance Comparison

In [ ]:
# Calculate Mean Squared Error for reconstruction quality
ae_recon = autoencoder.predict(x_test, batch_size=256, verbose=0)

z_mean, _, z = vae_encoder.predict(x_test, batch_size=256, verbose=0)
vae_recon = vae_decoder.predict(z, batch_size=256, verbose=0)

ae_mse = np.mean((x_test - ae_recon) ** 2)
vae_mse = np.mean((x_test - vae_recon) ** 2)

print("Autoencoder Reconstruction MSE:", ae_mse)
print("VAE Reconstruction MSE:", vae_mse)

models = ["Autoencoder", "VAE"]
mse_values = [ae_mse, vae_mse]

plt.figure(figsize=(6, 4))
plt.bar(models, mse_values)
plt.ylabel("Mean Squared Error")
plt.title("Reconstruction Error Comparison")
plt.show()


## Result and Conclusion

- The **Autoencoder** learns a compact representation of the input image and generally provides good image reconstruction.
- The **VAE** learns a probabilistic latent space using reconstruction loss and KL-divergence loss.
- The **VAE can generate new images** by sampling random latent vectors, while a basic Autoencoder is mainly used for reconstruction.
- Reconstruction quality can be compared using **MSE/loss**, while generated-image quality can be observed visually.

**Conclusion:** Autoencoders are effective for reconstruction, whereas VAEs provide both reconstruction and meaningful image generation capabilities.
